In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import torchvision.transforms as T

In [ ]:
# constants
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
DATASET_DIR = '/kaggle/input/datasets/hamzabinbutt/isic-challenge-datasets'
SAVE_DIR = '/kaggle/working'
IMG_SIZE = 380
BATCH_SIZE = 16
NUM_CLASSES = 11
D_MODEL = 256
DROPOUT = 0.3
CLASS_COLS = ['AKIEC','BCC','BEN_OTH','BKL','DF', 'INF','MAL_OTH','MEL','NV','SCCKA','VASC']

In [ ]:
ground_truth = pd.read_csv(os.path.join(DATASET_DIR, 'MILK10k_Training_GroundTruth.csv'))
training_meta_data = pd.read_csv(os.path.join(DATASET_DIR, 'MILK10k_Training_Metadata.csv'))
training_meta_data_copy = training_meta_data.drop(['image_type', 'attribution', 'copyright_license', 'image_manipulation'], axis=1)

cat_attrs = ['sex', 'skin_tone_class', 'site']
num_attrs = ['MONET_ulceration_crust', 'MONET_hair', 'MONET_vasculature_vessels', 'MONET_erythema', 'MONET_pigmented', 'MONET_gel_water_drop_fluid_dermoscopy_liquid', 'MONET_skin_markings_pen_ink_purple_pen']
age_index = [training_meta_data_copy.columns.get_loc('age_approx')]
cat_index = [training_meta_data_copy.columns.get_loc(c) for c in cat_attrs]
num_index = [training_meta_data_copy.columns.get_loc(c) for c in num_attrs]
print('Metadata shape:', training_meta_data_copy.shape)

In [ ]:
ground_truth_copy = ground_truth.copy()
ground_truth_copy['ground_truth'] = ground_truth_copy[CLASS_COLS].idxmax(axis=1)
ground_truth_splitter = ground_truth_copy[['lesion_id', 'ground_truth']]

X_train, X_val = train_test_split(ground_truth_splitter, stratify=ground_truth_splitter['ground_truth'], test_size=0.15, random_state=42)
train_ids = set(X_train['lesion_id'].values)
val_ids = set(X_val['lesion_id'].values)
print(f'Train: {len(train_ids)}  Val: {len(val_ids)}')

In [ ]:
train_df_split = training_meta_data_copy[training_meta_data_copy['lesion_id'].isin(train_ids)]

age_pipeline = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
num_pipeline = make_pipeline(SimpleImputer(strategy='mean'))
cat_pipeline = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore', sparse_output=False))
cat_pipeline.fit(training_meta_data_copy[cat_attrs])  # fit on full vocab

preprocessing = ColumnTransformer([('age_approx', age_pipeline, age_index), ('cat', cat_pipeline, cat_index), ('num', num_pipeline, num_index)], remainder='drop')
preprocessing.fit(train_df_split.to_numpy())

sample_row = training_meta_data_copy.iloc[[0]].to_numpy()
META_DIM = preprocessing.transform(sample_row).shape[1]
print(f'META_DIM after preprocessing: {META_DIM}')

In [ ]:
def load_dataset(base_dir: str):
    train_labels, train_data = [], []
    val_labels, val_data   = [], []
    skip = {'licenses'}
    for entry in sorted(os.scandir(base_dir), key=lambda x: x.name):
        if not entry.is_dir() or entry.name in skip:
            continue
        lesion_id = entry.name
        label_row = ground_truth[ground_truth['lesion_id'] == lesion_id].values
        if len(label_row) == 0:
            continue
        label = label_row.flatten()[1:].astype(np.float32)

        clin_path = derm_path = clin_meta = derm_meta = None
        for img_entry in sorted(os.scandir(entry.path), key=lambda x: x.name):
            isic_id = img_entry.name.removesuffix('.jpg')
            row = training_meta_data[training_meta_data['isic_id'] == isic_id]
            if len(row) == 0:
                continue
            image_type = row['image_type'].values[0]
            meta_row   = training_meta_data_copy[training_meta_data_copy['isic_id'] == isic_id].to_numpy()
            meta_vec = preprocessing.transform(meta_row).flatten().astype(np.float32)
            if image_type == 'dermoscopic':
                derm_path, derm_meta = img_entry.path, meta_vec
            else:
                clin_path, clin_meta = img_entry.path, meta_vec

        if lesion_id in train_ids:
            train_data.append((derm_path, clin_path, derm_meta, clin_meta))
            train_labels.append(label)
        else:
            val_data.append((derm_path, clin_path, derm_meta, clin_meta))
            val_labels.append(label)

    return train_data, train_labels, val_data, val_labels


print('Loading training data…')
train_data, train_labels, val_data, val_labels = load_dataset(os.path.join(DATASET_DIR, 'MILK10k_Training_Input', 'MILK10k_Training_Input'))
print(f'Train samples: {len(train_data)}  Val samples: {len(val_data)}')

In [ ]:
labels_arr = np.stack(train_labels) # (N, 11)
class_counts = labels_arr.sum(axis=0) # (11,)

total_samples = len(labels_arr)
neg_counts = total_samples - class_counts
pos_weight = neg_counts / np.maximum(class_counts, 1)
pos_weight = pos_weight / pos_weight.mean()
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)

print('Class weights (pos_weight):')
for c, cnt, w in zip(CLASS_COLS, class_counts.astype(int), pos_weight):
    print(f'  {c:30s}: count={cnt:5d}  pos_weight={w:.4f}')

# WRS: each lesion weight = inverse-freq of its rarest positive class
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = []
for row in labels_arr:
    pos = row > 0.5
    w = class_weights[pos].max() if pos.any() else class_weights.min()
    sample_weights.append(w)
sample_weights = torch.tensor(sample_weights, dtype=torch.double)
print(f'\nSample weight range: [{sample_weights.min():.6f}, {sample_weights.max():.6f}]')

In [ ]:
def get_transforms(train: bool = True):
    if train:
        return T.Compose([
            T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(
                brightness=0.3,
                contrast=0.3,
                saturation=0.2,   
                hue=0.05,
            ),
            T.RandomRotation(15),
            T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.2),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


class MilkDataset(Dataset):
    """One item = one lesion: (derm_img, clin_img, derm_meta, clin_meta, label). A black image is substituted when a modality path is None."""
    def __init__(self, data, labels, transform=None):
        self.data      = data
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def _load_img(self, path):
        if path is None:
            img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
        else:
            img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        derm_path, clin_path, derm_meta, clin_meta = self.data[idx]
        derm_img = self._load_img(derm_path)
        clin_img = self._load_img(clin_path)

        if derm_meta is None:
            derm_meta = np.zeros(META_DIM, dtype=np.float32)
        if clin_meta is None:
            clin_meta = np.zeros(META_DIM, dtype=np.float32)

        derm_meta = torch.tensor(derm_meta, dtype=torch.float32)
        clin_meta = torch.tensor(clin_meta, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return derm_img, clin_img, derm_meta, clin_meta, label

In [ ]:
train_ds = MilkDataset(train_data, train_labels, transform=get_transforms(True))
val_ds = MilkDataset(val_data, val_labels, transform=get_transforms(False))

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_ds), replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p = torch.sigmoid(logits)
        # Probability shift on negatives — reduces gradient from easy negatives
        p_neg = torch.clamp(p - self.clip, min=0.0, max=1.0)
        # Cross-entropy terms
        loss_pos = targets       * torch.log(p.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log((1 - p_neg).clamp(min=self.eps))
        # Focal weights — paper uses p^gamma_neg on negative term (not complement)
        focal_pos = (1 - p) ** self.gamma_pos
        focal_neg = p_neg ** self.gamma_neg  
        loss = -(focal_pos * loss_pos + focal_neg * loss_neg)
        return loss.mean()

criterion = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=0.0, clip=0.05)

In [ ]:
class GMU(nn.Module):
    """Gated Multimodal Unit"""
    def __init__(self, dim: int):
        super().__init__()
        self.h1 = nn.Linear(dim, dim)
        self.h2 = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim * 2, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x1, x2):
        h1 = torch.tanh(self.h1(x1))
        h2 = torch.tanh(self.h2(x2))
        z = torch.sigmoid(self.gate(torch.cat([x1, x2], dim=-1)))
        out = z * h1 + (1.0 - z) * h2   # pure gated blend — no extra residual
        return self.norm(out)


class MilkNet(nn.Module):
    """
    Multimodal skin lesion classifier.

    Architecture
    ------------
    - Shared EfficientNetV2-S backbone  → image embedding per view
    - Metadata MLP with residual        → metadata embedding per view
    - GMU (derm): fuses derm_img   + derm_meta
    - GMU (clin): fuses clin_img   + clin_meta
    - GMU (view): fuses derm_fused + clin_fused
    - Classifier head on concatenation of all three fused reps
    - Auxiliary heads on derm_fused and clin_fused for deep supervision
    """

    def __init__(self, num_classes: int=11, d_model: int=256, meta_dim: int=23, dropout: float=0.3):
        super().__init__()
        self._freeze_bn = False 

        # backbone
        self.backbone = timm.create_model('tf_efficientnetv2_s.in21k_ft_in1k', pretrained=True, num_classes=0, global_pool='avg')
        backbone_dim = self.backbone.num_features

        # image
        self.img_proj = nn.Sequential(
            nn.Linear(backbone_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.img_proj_residual = nn.Linear(backbone_dim, d_model, bias=False)

        # metadata mlp
        self.meta_enc = nn.Sequential(
            nn.Linear(meta_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, d_model),
            nn.LayerNorm(d_model),
        )
        self.meta_residual = nn.Linear(meta_dim, d_model, bias=False)

        # GMU fusion layers
        self.lgmu = GMU(d_model) # fuse image and metadata
        self.view_fusion = GMU(d_model) # derm_fused + clin_fused

        # auxiliary heads
        self.derm_head = nn.Linear(d_model, num_classes)
        self.clin_head = nn.Linear(d_model, num_classes)

        # main classifier head
        # input: cat(derm_fused, clin_fused, view_repr) = 3 × d_model
        self.head = nn.Sequential(
            nn.Linear(d_model * 3, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    # fix batcn normalization layers
    def train(self, mode: bool = True):
        """Override so that BN layers stay in eval() when _freeze_bn is set."""
        super().train(mode)
        if self._freeze_bn:
            for m in self.backbone.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                    m.eval()
        return self

    def freeze_bn(self):
        """Call once at start of Phase 2 to permanently freeze BN stats."""
        self._freeze_bn = True
        self.train()   

    # encoders
    def encode_img(self, x):
        raw = self.backbone(x)
        return self.img_proj(raw) + self.img_proj_residual(raw)

    def encode_meta(self, meta):
        return self.meta_enc(meta) + self.meta_residual(meta)

    # forward pass
    def forward(self, derm_img, clin_img, derm_meta, clin_meta):
        d_img  = self.encode_img(derm_img) # (B, d_model)
        c_img  = self.encode_img(clin_img)

        d_meta = self.encode_meta(derm_meta) # (B, d_model)
        c_meta = self.encode_meta(clin_meta)

        d_fused   = self.lgmu(d_img,  d_meta) # (B, d_model)
        c_fused   = self.lgmu(c_img,  c_meta)
        view_repr = self.view_fusion(d_fused, c_fused)

        # concatenate all three representations for the main head
        combined = torch.cat([d_fused, c_fused, view_repr], dim=-1)  # (B, 3*d)

        return {
            'fusion': self.head(combined),
            'derm':   self.derm_head(d_fused),
            'clin':   self.clin_head(c_fused),
        }


model = MilkNet(num_classes=NUM_CLASSES, d_model=D_MODEL, meta_dim=META_DIM, dropout=DROPOUT).to(DEVICE)
print(f'Backbone features: {model.backbone.num_features}')
print(f'Total params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
def compute_metrics(all_labels: np.ndarray, all_probs:  np.ndarray, print_per_class: bool = False) -> float:
    """Compute macro-F1 at the fixed ≥0.5 threshold used by the ISIC challenge leaderboard."""
    preds_bin = (all_probs >= 0.5).astype(int)
    labels_bin = (all_labels >= 0.5).astype(int)

    per_class = f1_score(labels_bin, preds_bin, average=None, zero_division=0)
    macro_f1 = per_class.mean()

    if print_per_class:
        print('Per-class F1 @ 0.5:')
        for name, f in zip(CLASS_COLS, per_class):
            bar = '█' * int(f * 20)
            print(f'{name:10s}: {f:.3f}  {bar}')

    return float(macro_f1)

In [ ]:
def run_epoch(loader, train: bool=True, scaler=None, optimizer=None, print_per_class: bool=False):
    model.train(train)

    total_loss = 0.0
    all_probs  = []
    all_labels = []

    for derm_img, clin_img, derm_meta, clin_meta, labels in loader:
        derm_img = derm_img.to(DEVICE)
        clin_img = clin_img.to(DEVICE)
        derm_meta = derm_meta.to(DEVICE)
        clin_meta = clin_meta.to(DEVICE)
        labels = labels.to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast('cuda'):
                preds = model(derm_img, clin_img, derm_meta, clin_meta)

                fusion_loss = criterion(preds['fusion'], labels)
                derm_loss = criterion(preds['derm'],   labels)
                clin_loss = criterion(preds['clin'],   labels)
                loss = fusion_loss + 0.4 * derm_loss + 0.4 * clin_loss

        if train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        with torch.no_grad():
            all_probs.append(torch.sigmoid(preds['fusion']).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    macro_f1 = compute_metrics(all_labels, all_probs, print_per_class=print_per_class)

    return total_loss / len(loader.dataset), macro_f1

In [ ]:
model.backbone.set_grad_checkpointing(True)
for p in model.backbone.parameters():
    p.requires_grad = False

PHASE1_EPOCHS = 15
optimizer1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
# Warm-up for 3 epochs, then cosine decay
def lr_lambda(epoch):
    warmup = 3
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(PHASE1_EPOCHS - warmup, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler1 = torch.optim.lr_scheduler.LambdaLR(optimizer1, lr_lambda=lr_lambda)
scaler1 = torch.amp.GradScaler('cuda')

best_val_f1 = 0.0
best_ckpt = os.path.join(SAVE_DIR, 'milk10k_pt_gmu_best.pt')

print('-- Phase 1: frozen backbone --')
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True,  scaler=scaler1, optimizer=optimizer1)
    show_detail    = (epoch % 5 == 0) or (epoch == PHASE1_EPOCHS)
    va_loss, va_f1 = run_epoch(val_loader, train=False, print_per_class=show_detail)
    scheduler1.step()

    flag = ''
    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        torch.save(model.state_dict(), best_ckpt)
        flag = ' <- best'

    print(
        f'Ep {epoch:02d}/{PHASE1_EPOCHS} '
        f'| tr_loss {tr_loss:.4f} tr_f1 {tr_f1:.4f} '
        f'| va_loss {va_loss:.4f} va_f1 {va_f1:.4f}{flag}'
    )


In [ ]:
# Load best Phase-1 checkpoint before unfreezing
model.load_state_dict(torch.load(best_ckpt))

# freeze entire backbone, then selectively unfreeze later blocks
for p in model.backbone.parameters():
    p.requires_grad = False

for block in model.backbone.blocks[3:]:
    for p in block.parameters():
        p.requires_grad = True
for p in model.backbone.conv_head.parameters():
    p.requires_grad = True

# permanently freeze BN statistics via the train() override
model.freeze_bn()
backbone_trainable = [p for p in model.backbone.parameters() if p.requires_grad]

HEAD_LR = 3e-5
BACKBONE_LR = HEAD_LR / 10

optimizer2 = torch.optim.AdamW([
    {'params': backbone_trainable, 'lr': BACKBONE_LR},
    {'params': model.img_proj.parameters(), 'lr': HEAD_LR},
    {'params': model.img_proj_residual.parameters(),'lr': HEAD_LR},
    {'params': model.meta_enc.parameters(), 'lr': HEAD_LR},
    {'params': model.meta_residual.parameters(), 'lr': HEAD_LR},
    {'params': model.lgmu.parameters(), 'lr': HEAD_LR},
    {'params': model.view_fusion.parameters(), 'lr': HEAD_LR},
    {'params': model.derm_head.parameters(), 'lr': HEAD_LR},
    {'params': model.clin_head.parameters(), 'lr': HEAD_LR},
    {'params': model.head.parameters(), 'lr': HEAD_LR},
], weight_decay=1e-4)

PHASE2_EPOCHS = 40
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=PHASE2_EPOCHS, eta_min=1e-7)
scaler2 = torch.amp.GradScaler('cuda')

print('\n-- Phase 2: fine-tune (discriminative LR) --')
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True,  scaler=scaler2, optimizer=optimizer2)
    show_detail = (epoch % 5 == 0) or (epoch == PHASE2_EPOCHS)
    va_loss, va_f1 = run_epoch(val_loader, train=False, print_per_class=show_detail)
    scheduler2.step()

    flag = ''
    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        torch.save(model.state_dict(), best_ckpt)
        flag = '<- best'

    print(
        f'Ep {epoch:02d}/{PHASE2_EPOCHS} '
        f'| tr_loss {tr_loss:.4f} tr_f1 {tr_f1:.4f} '
        f'| va_loss {va_loss:.4f} va_f1 {va_f1:.4f}{flag}'
    )

print(f'\nBest val macro-F1: {best_val_f1:.4f}')

In [ ]:
model.load_state_dict(torch.load(best_ckpt))
model.eval()

test_meta_data = pd.read_csv(os.path.join(DATASET_DIR, 'MILK10k_Test_Metadata.csv'))
test_meta_data_copy = test_meta_data.drop(['image_type', 'attribution', 'copyright_license', 'image_manipulation'], axis=1)
test_ids = []
test_data = []

for entry in sorted(os.scandir(os.path.join(DATASET_DIR, 'MILK10k_Test_Input', 'MILK10k_Test_Input')), key=lambda x: x.name):
    if not entry.is_dir() or entry.name == 'licenses':
        continue
    lesion_id = entry.name
    test_ids.append(lesion_id)

    clin_path = derm_path = clin_meta = derm_meta = None
    for img_entry in sorted(os.scandir(entry.path), key=lambda x: x.name):
        isic_id = img_entry.name.removesuffix('.jpg')
        image_type = test_meta_data[test_meta_data['isic_id'] == isic_id]['image_type'].values[0]
        meta_row = test_meta_data_copy[test_meta_data_copy['isic_id'] == isic_id].to_numpy()
        meta_vec = preprocessing.transform(meta_row).flatten().astype(np.float32)
        if image_type == 'dermoscopic':
            derm_path, derm_meta = img_entry.path, meta_vec
        else:
            clin_path, clin_meta = img_entry.path, meta_vec
    test_data.append((derm_path, clin_path, derm_meta, clin_meta))

print(f'Test samples: {len(test_ids)}')

In [ ]:
dummy_labels = [np.zeros(NUM_CLASSES, dtype=np.float32)] * len(test_data)
test_ds = MilkDataset(test_data, dummy_labels, transform=get_transforms(False))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

all_preds = []
with torch.no_grad():
    for derm_img, clin_img, derm_meta, clin_meta, _ in test_loader:
        with torch.amp.autocast('cuda'):
            preds = model(derm_img.to(DEVICE), clin_img.to(DEVICE), derm_meta.to(DEVICE), clin_meta.to(DEVICE))
        probs = torch.sigmoid(preds['fusion']).cpu().numpy()
        all_preds.append(probs)

pred_arr = np.vstack(all_preds)
print('Predictions shape:', pred_arr.shape)

In [ ]:
df_res = pd.DataFrame(pred_arr, columns=CLASS_COLS)
df_res.insert(0, 'image', test_ids)
out_csv = os.path.join(SAVE_DIR, 'milk10k_pt_gmu_submission.csv')
df_res.to_csv(out_csv, index=False)
print('Saved:', out_csv)
print(df_res.head())